# Historico del Nivel del Rio Uruguai como Traget

In [0]:
df = spark.read.csv("/Volumes/weather/raw/ana_volume/historical/74100000_Cotas.csv", header=True, inferSchema=True)
display(df)

In [0]:
from pyspark.sql.functions import col, lit, concat_ws, lpad, expr, current_timestamp
from pyspark.sql import functions as F

csv_path = "/Volumes/weather/raw/ana_volume/historical/74100000_Cotas.csv"
json_path = "/Volumes/weather/raw/ana_volume/json/Histo/histo_serie_74100000.json"

# lectura
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

# columnas de cotas
cota_cols = [f"Cota{str(i).zfill(2)}" for i in range(1, 32)]

# unpivot
stack_expr = "stack({0}, {1}) as (dia, cota)".format(
    len(cota_cols),
    ",".join([f"'{i+1}', {c}" for i, c in enumerate(cota_cols)])
)

df_long = df.select(
    col("EstacaoCodigo").alias("codigoestacao"),
    col("Data"),
    expr(stack_expr)
)

# separar mes y año
df_dates = (
    df_long
    .withColumn("mes", F.split(col("Data"), "/")[1])
    .withColumn("anio", F.split(col("Data"), "/")[2])
)

# construir fecha con hora 12:00 usando expr for try_cast to tolerate fechas inválidas
df_dates = df_dates.withColumn(
    "Data_Hora_Medicao",
    expr(
        "try_cast(concat(anio, '-', mes, '-', lpad(dia, 2, '0'), ' 12:00:00') as timestamp)"
    )
)

# construir json final
df_final = (
    df_dates
    .filter(col("cota").isNotNull())
    .select(
        lit(None).alias("Chuva_Adotada"),
        lit("0").alias("Chuva_Adotada_Status"),
        col("cota").cast("string").alias("Cota_Adotada"),
        lit("0").alias("Cota_Adotada_Status"),
        current_timestamp().alias("Data_Atualizacao"),
        col("Data_Hora_Medicao"),
        lit(None).alias("Vazao_Adotada"),
        lit("0").alias("Vazao_Adotada_Status"),
        col("codigoestacao").cast("string")
    )
)

# escritura json
(
    df_final
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(json_path)
)
